AI Acknowledgement is provided in a separate document, including prompts and responses used to build this notebook.

In [28]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import norm

DATA = pd.read_csv("fund_returns.csv")
DATA

,fund,date,excess_ret,mkt_rf
0,F01,2012-01,3.1672,5.05
1,F01,2012-02,3.0904,4.42
2,F01,2012-03,2.6921,3.11
3,F01,2012-04,0.9631,-0.85
4,F01,2012-05,-8.0615,-6.19
...,...,...,...,...
4124,F40,2016-08,-2.0036,0.50
4125,F40,2016-09,1.1491,0.25
4126,F40,2016-10,0.9566,-2.02
4127,F40,2016-11,10.5612,4.86


# Question 1

## a)


(6 marks) For each fund, regress excess_ret on an intercept and mkt_rf by ordinary
least squares. Annualise: let 𝑎𝑘 be twelve times the estimated intercept and 𝑠𝑘 twelve times
its conventional OLS standard error. Call the fund with the largest 𝑎𝑘 the star. Report 𝑎𝑘
for the star.

In [31]:
def get_regression_results(data):
    fund_regression_results = {}

    # Regress each fund's excess returns on the market returns
    for fund in data["fund"].unique():
        fund_data = data[data["fund"] == fund][["excess_ret", "mkt_rf"]]
        fund_regression_results[fund] = sm.OLS(fund_data["excess_ret"], sm.add_constant(fund_data["mkt_rf"])).fit()

    return fund_regression_results

def get_star_fund(regression_results):
    # Find the fund with the highest alpha
    star_fund = max(regression_results.items(), key=lambda x: x[1].params["const"])
    return star_fund[0]

def annualize_alpha(alpha, periods_per_year=12):
    return (1 + alpha) ** periods_per_year - 1

fund_regression_results = get_regression_results(DATA)
star_fund = get_star_fund(fund_regression_results)
star_model = fund_regression_results[star_fund]

a_k = 12 * star_model.params["const"]
s_k = 12 * star_model.bse["const"]
print(f"The star fund is {star_fund} with an annualised alpha of {a_k:.4f} (standard error: {s_k:.4f}).")

The star fund is F21 with an annualised alpha of 3.9373 (standard error: 2.1592).


## b)


**(6 marks) Analyse the star alone.** With a flat prior on $\alpha_k$ and the approximation

$$
a_k \mid \alpha_k \sim \mathcal{N}(\alpha_k, s_k^2),
$$

the posterior is

$$
\alpha_k \mid a_k \sim \mathcal{N}(a_k, s_k^2).
$$

Report $\mathbb{P}(\alpha_k > 0 \mid a_k)$.

In [34]:
# Posterior probability that the star's true alpha is positive
prob_positive = norm.cdf(a_k / s_k)

print(f"P(alpha > 0 | a) = {prob_positive:.4f} ({prob_positive:.2%})")

P(alpha > 0 | a) = 0.9659 (96.59%)
